# NB_hospital_daily_summary

Builds `<catalog>.gold.hospital_daily_summary` as defined in `sttm_silver_to_gold.xlsx`, sheet `hospital_daily_summary`.

* **Grain:** one row per hospital per calendar day
* **Driving table:** `<catalog>.silver.lab_results`, grouped by `hospital_id` and `test_date`
* **Joins (LEFT):** `silver.hospitals` on `hospital_id`, `silver.calendar` on `date_key = date_format(test_date, 'yyyyMMdd')`
* **Window:** `LAG(total_tests)` by `hospital_id` ordered by `summary_date`, for the day-over-day trend
* **Filters:** none
* **Load type:** FULL, using `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
dbutils.widgets.text("catalog", "adb_caresync")
catalog = dbutils.widgets.get("catalog").strip()

src = f"{catalog}.silver"
target_table = f"{catalog}.gold.hospital_daily_summary"
print(f"Source schema: {src} | Target: {target_table}")

In [0]:
# Columns each source table must have, per the STTM column mapping
required = {
    "lab_results": ["lab_result_id", "hospital_id", "patient_id", "doctor_id", "result_status", "test_date"],
    "hospitals": ["hospital_id", "hospital_name"],
    "calendar": ["date_key", "day_name", "is_weekend"],
}

problems = []
for tbl, cols in required.items():
    fqn = f"{src}.{tbl}"
    if not spark.catalog.tableExists(fqn):
        problems.append(f"Missing table: {fqn}")
        continue
    existing = set(spark.table(fqn).columns)
    missing = [c for c in cols if c not in existing]
    if missing:
        problems.append(f"{fqn} is missing columns: {missing}")

if problems:
    raise Exception("Source validation failed:\n" + "\n".join(problems))
print("All source tables and columns present.")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per hospital per day with lab test volume, quality rates and day-over-day trend (STTM hospital_daily_summary)'
AS
WITH daily AS (
  SELECT
    hospital_id,
    CAST(test_date AS DATE)                 AS summary_date,
    COUNT(*)                                AS total_tests,
    COUNT(DISTINCT patient_id)              AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)               AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')    AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')    AS critical_test_count
  FROM {src}.lab_results
  GROUP BY hospital_id, CAST(test_date AS DATE)
),
with_lag AS (
  SELECT
    *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)
SELECT
  CAST(d.hospital_id AS INT)                                                      AS hospital_id,
  CAST(h.hospital_name AS STRING)                                                 AS hospital_name,
  d.summary_date                                                                  AS summary_date,
  c.day_name                                                                      AS day_name,
  c.is_weekend                                                                    AS is_weekend,
  CAST(d.total_tests AS INT)                                                      AS total_tests,
  CAST(d.unique_patients_tested AS INT)                                           AS unique_patients_tested,
  CAST(d.active_doctor_count AS INT)                                              AS active_doctor_count,
  CAST(d.abnormal_test_count AS INT)                                              AS abnormal_test_count,
  CAST(d.critical_test_count AS INT)                                              AS critical_test_count,
  CAST(ROUND(d.abnormal_test_count * 100.0 / NULLIF(d.total_tests, 0), 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(ROUND(d.critical_test_count * 100.0 / NULLIF(d.total_tests, 0), 2) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(d.prev_day_total_tests AS INT)                                             AS prev_day_total_tests,
  CASE
    WHEN d.prev_day_total_tests IS NULL OR d.prev_day_total_tests = 0 THEN NULL
    ELSE CAST(ROUND((d.total_tests - d.prev_day_total_tests) * 100.0 / d.prev_day_total_tests, 2) AS DECIMAL(6,2))
  END                                                                             AS test_volume_change_pct
FROM with_lag d
LEFT JOIN {src}.hospitals h
  ON d.hospital_id = h.hospital_id
LEFT JOIN {src}.calendar c
  ON c.date_key = CAST(date_format(d.summary_date, 'yyyyMMdd') AS INT)
""")
print(f"Rebuilt {target_table}")

In [0]:
record_count = spark.table(target_table).count()
print(f"{target_table} record count: {record_count}")
dbutils.notebook.exit(str(record_count))